01 — Load the Caravan data
Read the three original basin CSVs and keep the columns used by this experiment.

In [2]:
from pathlib import Path
import tarfile
import urllib.request

# Your project folder.
ROOT = Path(
    r"C:\Users\Pranav Yeturu\Desktop\lab2\Early-Flash-Flood-Prediction"
)

# Create the folder where the dataset will be saved.
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# Official Caravan archive.
URL = (
    "https://zenodo.org/api/records/15530022/"
    "files/Caravan-csv.tar.gz/content"
)

# Files required for our project.
needed = {
    "attributes_other_camels.csv",
    "license_camels.md",
    "camels_01411300.csv",
    "camels_05495500.csv",
    "camels_14182500.csv",
}

# Skip files that already exist.
needed = {
    name for name in needed
    if not (RAW / name).exists()
}

if not needed:
    print("All required files already exist.")

else:
    print("Downloading the dataset subset. Please wait several minutes.")

    # Read the compressed archive without saving the entire archive.
    with urllib.request.urlopen(URL, timeout=120) as response:
        with tarfile.open(fileobj=response, mode="r|gz") as archive:

            for item in archive:
                name = Path(item.name).name

                if item.isfile() and name in needed:
                    contents = archive.extractfile(item).read()

                    # Write to a temporary file first so an interrupted
                    # write is not mistaken for a completed download.
                    destination = RAW / name
                    temporary = RAW / (name + ".part")
                    temporary.write_bytes(contents)
                    temporary.replace(destination)

                    print("Downloaded:", name)
                    needed.remove(name)

                    # Stop once all five files are downloaded.
                    if not needed:
                        break

    if needed:
        raise RuntimeError(f"Files not found in the archive: {needed}")

    print("Download complete!")

print("Dataset location:", RAW)

Downloaded: attributes_other_camels.csv
Downloaded: license_camels.md
Downloaded: camels_05495500.csv
Downloaded: camels_14182500.csv
Downloaded: camels_01411300.csv
Download complete!
Dataset location: C:\Users\Pranav Yeturu\Desktop\lab2\Early-Flash-Flood-Prediction\data\raw


In [3]:
from pathlib import Path
import pandas as pd

# VS Code may start in the project root or in the notebook folder.
ROOT = Path.cwd()
if not (ROOT / "data" / "raw").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "data" / "raw").is_dir():
    raise FileNotFoundError("Open the project containing data/raw, and download the subset first.")
print("Project folder:", ROOT)
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

basins = ["camels_01411300", "camels_05495500", "camels_14182500"]
tables = []

Project folder: c:\Users\Pranav Yeturu\Desktop\lab2\Early-Flash-Flood-Prediction


Load each basin
One row is a basin-day, not a separate gauge. The original files have 41 columns; the saved table keeps date, discharge, rainfall, temperature, and basin ID.

In [4]:
for basin in basins:
    df = pd.read_csv(RAW / f"{basin}.csv", parse_dates=["date"])
    print(f"\n{basin}: {df.shape[0]} rows, {df.shape[1]} columns")
    print("Original columns:", df.columns.tolist())

    # Keep three measured variables plus the date and basin identifier.
    df = df[["date", "streamflow", "total_precipitation_sum", "temperature_2m_mean"]].copy()
    df = df.rename(columns={"total_precipitation_sum": "rainfall",
                            "temperature_2m_mean": "temperature"})
    df["basin"] = basin
    tables.append(df)


camels_01411300: 26662 rows, 41 columns
Original columns: ['date', 'dewpoint_temperature_2m_max', 'dewpoint_temperature_2m_mean', 'dewpoint_temperature_2m_min', 'potential_evaporation_sum_ERA5_LAND', 'potential_evaporation_sum_FAO_PENMAN_MONTEITH', 'snow_depth_water_equivalent_max', 'snow_depth_water_equivalent_mean', 'snow_depth_water_equivalent_min', 'streamflow', 'surface_net_solar_radiation_max', 'surface_net_solar_radiation_mean', 'surface_net_solar_radiation_min', 'surface_net_thermal_radiation_max', 'surface_net_thermal_radiation_mean', 'surface_net_thermal_radiation_min', 'surface_pressure_max', 'surface_pressure_mean', 'surface_pressure_min', 'temperature_2m_max', 'temperature_2m_mean', 'temperature_2m_min', 'total_precipitation_sum', 'u_component_of_wind_10m_max', 'u_component_of_wind_10m_mean', 'u_component_of_wind_10m_min', 'v_component_of_wind_10m_max', 'v_component_of_wind_10m_mean', 'v_component_of_wind_10m_min', 'volumetric_soil_water_layer_1_max', 'volumetric_soil_wat

Save a combined table

In [5]:
data = pd.concat(tables, ignore_index=True)
data.to_csv(PROCESSED / "caravan_selected.csv", index=False)
print("\nSaved selected data:", data.shape)


Saved selected data: (79986, 5)


Inspecting Output

In [6]:
display(data.head())
display(data.groupby("basin").size().rename("daily_rows").to_frame())

,date,streamflow,rainfall,temperature,basin
0,1951-01-01,NaN,0.00,0.37,camels_01411300
1,1951-01-02,NaN,0.00,2.68,camels_01411300
2,1951-01-03,NaN,0.57,10.05,camels_01411300
3,1951-01-04,NaN,2.54,11.01,camels_01411300
4,1951-01-05,NaN,0.00,3.35,camels_01411300


,daily_rows
basin,
camels_01411300,26662
camels_05495500,26662
camels_14182500,26662
